# Linear Regression

## What you'll learn
- How linear regression predicts a number from a weighted sum of features
- How to read the model's coefficients
- How to measure a regression model: MAE, RMSE, and R squared
- How to add a category column (segment) with one-hot encoding
- How to chart predictions against actual values

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

## The idea

Linear regression finds the weights that best fit

`prediction = intercept + w1 * feature1 + w2 * feature2 + ...`

Each weight (a **coefficient**) says how much the prediction changes when that feature
goes up by one, with the other features held still.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

features = ["spend_last_year", "years_as_customer", "on_contract", "support_tickets"]
X = df[features]
y = df["next_year_spend"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)

print(f"intercept: {model.intercept_:,.0f}")
for name, coef in zip(features, model.coef_):
    print(f"{name:20s} {coef:,.2f}")

Read it like this: each extra dollar of last year's spend adds about a dollar to next
year's prediction, and having a contract adds a fixed amount on top. A negative
coefficient on support tickets means more problems, lower expected spend.

## Measuring a regression model

| Metric | Meaning | Better |
|--------|---------|--------|
| MAE | Average size of the miss, in USD | Lower |
| RMSE | Like MAE, but punishes big misses more | Lower |
| R squared | Share of the variation the model explains (1.0 is perfect, 0 is no better than the average) | Higher |

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

predictions = model.predict(X_test)
print(f"MAE:  {mean_absolute_error(y_test, predictions):,.0f}")
print(f"RMSE: {root_mean_squared_error(y_test, predictions):,.0f}")
print(f"R2:   {r2_score(y_test, predictions):.3f}")

## Adding a category column

Models need numbers, so a text column like `segment` is turned into one 0/1 column per
value: **one-hot encoding**. `pd.get_dummies` does it in one line; `drop_first=True` drops
one column that the others already imply.

In [ ]:
X2 = pd.get_dummies(df[features + ["segment"]], columns=["segment"], drop_first=True, dtype=int)
print(X2.columns.tolist())

X2_train, X2_test, y_train, y_test = train_test_split(X2, y, test_size=0.2, random_state=42)
model2 = LinearRegression().fit(X2_train, y_train)
print(f"R2 with segment: {r2_score(y_test, model2.predict(X2_test)):.3f}")

## Predicted vs actual

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, predictions, alpha=0.4)
limit = max(y_test.max(), predictions.max())
ax.plot([0, limit], [0, limit], color="grey", linestyle="--")   # perfect-prediction line
ax.set_xlabel("Actual next-year spend (USD)")
ax.set_ylabel("Predicted (USD)")
ax.set_title("Linear regression: predicted vs actual")
plt.show()

Points close to the dashed line are good predictions. A pattern of points far off the line
in one area would show where the model struggles.

## Practice

1. Fit a model using only `spend_last_year`. What is its R squared?
2. Add `discount_pct` and `avg_days_to_pay` to the features. Does the test MAE improve?
3. Which coefficient in the segment model is largest? What does it mean in plain words?
4. Predict next-year spend for a new account: spend 250,000, 6 years, on contract,
   3 support tickets. (Hint: build a one-row DataFrame with the same columns as `X`.)

In [ ]:
# Your practice code here

## Summary

- Linear regression predicts a number as a weighted sum of features.
- Coefficients show each feature's effect, holding the others still.
- Measure with MAE, RMSE (lower is better), and R squared (higher is better), on the test set.
- One-hot encoding turns categories into 0/1 columns the model can use.
- A predicted-vs-actual chart shows where the model is right and wrong.

## Practice Solutions

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

y = df["next_year_spend"]

# 1.
X = df[["spend_last_year"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
one = LinearRegression().fit(X_train, y_train)
print(f"R2 with one feature: {r2_score(y_test, one.predict(X_test)):.3f}")

# 2.
base = ["spend_last_year", "years_as_customer", "on_contract", "support_tickets"]
for cols in [base, base + ["discount_pct", "avg_days_to_pay"]]:
    X_train, X_test, y_train, y_test = train_test_split(df[cols], y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(X_train, y_train)
    print(len(cols), "features, MAE:", f"{mean_absolute_error(y_test, m.predict(X_test)):,.0f}")

# 3.
X2 = pd.get_dummies(df[base + ["segment"]], columns=["segment"], drop_first=True, dtype=int)
m2 = LinearRegression().fit(X2, y)
coefs = pd.Series(m2.coef_, index=X2.columns)
print(coefs.abs().sort_values(ascending=False).head(3))
# Each coefficient is the USD change for a one-unit increase. For a 0/1 column such as
# segment_Distributor, it is the difference versus the segment that was dropped (Clinic).

# 4.
final = LinearRegression().fit(df[base], y)
new_account = pd.DataFrame([{"spend_last_year": 250_000, "years_as_customer": 6,
                             "on_contract": 1, "support_tickets": 3}])
print(f"Predicted next-year spend: {final.predict(new_account)[0]:,.0f} USD")